# CIFAR-10 STAQ

A notebook for the CIFAR-10 STAQ workflow:

- load config, CLIP, concepts, and data
- load or train Concept-QA
- load or train baseline and STAQ
- optionally sweep `lambda_adv` (or later `alpha_sens`) and save paper-style plots
- probe the sensitive concept set before retraining
- check sample-level sensitive-label sanity
- replay tiny-start cases

In [ ]:
%pip install -e ..

In [ ]:
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path

import torch

from staq.analysis import (
    plot_hparam_sweep_summary,
    plot_rollout_comparisons,
    plot_training_curves,
    sample_intuition_replays,
    summarize_hparam_sweep,
)
from staq.config import Cifar10StaqConfig, default_paths
from staq.core import (
    build_concept_dictionary,
    concept_answers_batch,
    load_clip_model,
    load_concept_qa_checkpoint,
    load_concepts,
    load_run_bundle,
    make_sensitive_mask,
    save_bundle_checkpoint,
)
from staq.data import get_cifar10_datasets, get_cifar10_loaders, get_raw_cifar10_dataset
from staq.models import ConceptNet2
from staq.sensitive_labels import (
    build_cifar10_sensitive_match,
    build_sensitive_labels,
    load_sensitive_labels,
    save_sensitive_labels,
)
from staq.training import HistorySamplingConfig, build_staq_models, fit_concept_qa, fit_staq, seed_everything
from staq.training.concept_qa import load_gpt_answers

In [ ]:
repo_root = Path.cwd().resolve()
if not (repo_root / "staq").exists() and (repo_root.parent / "staq").exists():
    repo_root = repo_root.parent

paths = default_paths(repo_root=repo_root)
paths.ensure_artifact_dirs()

config = Cifar10StaqConfig()
device = config.device
seed_everything(config.random_seed)

print(f"repo_root: {repo_root}")
print(f"artifacts_root: {paths.artifacts_root}")
print(f"device: {device}")

In [ ]:
model_clip, preprocess = load_clip_model(config.clip_model_name, device=device)
concepts = load_concepts(paths.concept_file)
dictionary = build_concept_dictionary(model_clip=model_clip, concepts=concepts, device=device)
sensitive_match = build_cifar10_sensitive_match(concepts)
sens_idx = sensitive_match.indices
sensitive_mask = make_sensitive_mask(config.max_queries, sens_idx, device)

train_ds, test_ds = get_cifar10_datasets(transform=preprocess, root=paths.data_root)
train_loader, test_loader = get_cifar10_loaders(
    transform=preprocess,
    root=paths.data_root,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
)
raw_test_ds = get_raw_cifar10_dataset(paths.data_root, train=False)

print(f"# concepts: {len(concepts)}")
print(f"# sensitive concepts matched: {len(sensitive_match.matched)}")
print(sensitive_match.matched)
if sensitive_match.missing:
    print(f"# sensitive concepts missing: {len(sensitive_match.missing)}")
    print(sensitive_match.missing)

In [ ]:
qa_checkpoint = paths.checkpoints_root / "concept_qa_cifar10.pt"
qa_source = qa_checkpoint
if qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(qa_checkpoint, device=device)
elif paths.bootstrap_concept_qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(paths.bootstrap_concept_qa_checkpoint, device=device)
    qa_source = paths.bootstrap_concept_qa_checkpoint
elif paths.gpt_answers_file.exists():
    gpt_answers = load_gpt_answers(paths.gpt_answers_file)
    qa_model = ConceptNet2().to(device)
    qa_optimizer = torch.optim.SGD(qa_model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
    qa_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(qa_optimizer, T_max=200)
    qa_history = fit_concept_qa(
        model=qa_model,
        train_loader=train_loader,
        eval_loader=test_loader,
        optimizer=qa_optimizer,
        scheduler=qa_scheduler,
        num_epochs=2,
        model_clip=model_clip,
        dictionary=dictionary,
        gpt_answers=gpt_answers,
        clip_device=device,
        train_device=device,
    )
    torch.save(qa_model.state_dict(), qa_checkpoint)
    with open(paths.runs_root / "concept_qa_cifar10_history.json", "w", encoding="utf-8") as handle:
        json.dump(qa_history, handle, indent=2)
    answering_model = qa_model.eval()
else:
    raise FileNotFoundError("No local Concept-QA checkpoint, bootstrap checkpoint, or GPT answers file available.")

print(f"Concept-QA ready from: {qa_source}")

In [ ]:
sensitive_labels_dir = paths.sensitive_labels_root

label_files = [
    sensitive_labels_dir / "s_soft_train.npy",
    sensitive_labels_dir / "s_hard_train.npy",
    sensitive_labels_dir / "s_soft_test.npy",
    sensitive_labels_dir / "s_hard_test.npy",
]

if all(path.exists() for path in label_files):
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "cache"
else:
    s_soft_train, s_hard_train = build_sensitive_labels(
        loader=train_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        sens_idx=sens_idx,
        clip_device=device,
        tau=config.sensitive_tau,
        topk=config.sensitive_topk,
        desc="Building sensitive labels (train)",
    )
    s_soft_test, s_hard_test = build_sensitive_labels(
        loader=test_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        sens_idx=sens_idx,
        clip_device=device,
        tau=config.sensitive_tau,
        topk=config.sensitive_topk,
        desc="Building sensitive labels (test)",
    )
    save_sensitive_labels(
        sensitive_labels_dir,
        train_soft=s_soft_train,
        train_hard=s_hard_train,
        test_soft=s_soft_test,
        test_hard=s_hard_test,
    )
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "built_and_saved"

print(f"Sensitive labels ready from: {label_source} -> {sensitive_labels_dir}")
print(
    {
        "s_soft_train": sensitive_label_cache["s_soft_train"].shape,
        "s_hard_train": sensitive_label_cache["s_hard_train"].shape,
        "s_soft_test": sensitive_label_cache["s_soft_test"].shape,
        "s_hard_test": sensitive_label_cache["s_hard_test"].shape,
    }
)
print(
    "Hard-positive rate (train/test):",
    float(sensitive_label_cache["s_hard_train"].mean()),
    float(sensitive_label_cache["s_hard_test"].mean()),
)

In [ ]:
def load_or_train_bundle(
    run_name,
    lambda_adv,
    alpha_sens,
    min_history=config.min_history,
    max_history=config.max_history,
    non_sensitive_only=config.non_sensitive_history_only,
    epochs=2,
    learning_rate=config.learning_rate,
    max_train_batches=60 if device.type == "cpu" else None,
    max_test_batches=30 if device.type == "cpu" else None,
    force_retrain=False,
):
    ckpt_path = paths.checkpoints_root / f"{run_name}_best.pt"
    history_path = paths.runs_root / f"{run_name}_history.json"
    if ckpt_path.exists() and not force_retrain:
        return load_run_bundle(ckpt_path, device=device, max_queries=config.max_queries, num_classes=config.num_classes)

    actor_checkpoint = str(paths.bootstrap_actor_checkpoint) if paths.bootstrap_actor_checkpoint.exists() else None
    classifier_checkpoint = str(paths.bootstrap_classifier_checkpoint) if paths.bootstrap_classifier_checkpoint.exists() else None

    actor, classifier, s_head = build_staq_models(
        max_queries=config.max_queries,
        num_classes=config.num_classes,
        device=device,
        actor_eps=config.actor_eps,
        actor_checkpoint=actor_checkpoint,
        classifier_checkpoint=classifier_checkpoint,
    )
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(s_head.parameters()),
        lr=learning_rate,
    )
    history_config = HistorySamplingConfig(
        min_history=min_history,
        max_history=max_history,
        non_sensitive_only=non_sensitive_only,
    )
    history, best = fit_staq(
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        optimizer=optimizer,
        train_loader=train_loader,
        test_loader=test_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        sens_idx=sens_idx,
        history_config=history_config,
        clip_device=device,
        train_device=device,
        threshold_for_binarization=config.threshold_for_binarization,
        lambda_adv=lambda_adv,
        alpha_sens=alpha_sens,
        sensitive_tau=config.sensitive_tau,
        sensitive_topk=config.sensitive_topk,
        num_epochs=epochs,
        max_train_batches=max_train_batches,
        max_test_batches=max_test_batches,
    )
    save_bundle_checkpoint(
        checkpoint_path=ckpt_path,
        metadata={
            "run_name": run_name,
            "lambda_adv": lambda_adv,
            "alpha_sens": alpha_sens,
            "best_test_acc": best["test_acc"],
            "best_epoch": best["epoch"],
            "history_config": {
                "min_history": history_config.min_history,
                "max_history": history_config.max_history,
                "non_sensitive_only": history_config.non_sensitive_only,
            },
            "actor_state_dict": best["actor_state_dict"],
            "classifier_state_dict": best["classifier_state_dict"],
            "s_head_state_dict": best["s_head_state_dict"],
        },
    )
    with open(history_path, "w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)
    return load_run_bundle(ckpt_path, device=device, max_queries=config.max_queries, num_classes=config.num_classes)


def load_run_history(run_name):
    history_path = paths.runs_root / f"{run_name}_history.json"
    if not history_path.exists():
        raise FileNotFoundError(f"Missing run history: {history_path}")
    with open(history_path, "r", encoding="utf-8") as handle:
        return json.load(handle)


baseline_bundle = load_or_train_bundle("baseline", lambda_adv=0.0, alpha_sens=0.0, epochs=5)
staq_bundle = load_or_train_bundle("lam_0.80", lambda_adv=0.8, alpha_sens=0.0, epochs=5)

print(baseline_bundle["ckpt_path"])
print(staq_bundle["ckpt_path"])

def answer_builder(images):
    return concept_answers_batch(
        images=images,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        clip_device=device,
        train_device=device,
        threshold=config.threshold_for_binarization,
    )

In [ ]:
intuition_records = sample_intuition_replays(
    dataset=test_ds,
    answer_builder=answer_builder,
    baseline_bundle=baseline_bundle,
    staq_bundle=staq_bundle,
    concepts=concepts,
    sensitive_mask=sensitive_mask,
    class_names=raw_test_ds.classes,
    num_cases=6,
    pool_size=400 if device.type == "cpu" else 1500,
    num_trials=3,
    min_history=0,
    max_history=1,
    history_mode="non_sensitive",
    prefer_baseline_sensitive=True,
)

intuition_fig = plot_rollout_comparisons(
    records=intuition_records,
    raw_dataset=raw_test_ds,
    output_path=paths.figures_root / "cifar10_intuition_replay_examples.png",
    title_prefix="tiny-start replay",
)

print(f"Saved tiny-start replay figure: {intuition_fig}")

## Optional Sweep

Use this section for a small sweep with saved histories and paper-style plots.
For the current paper path, sweep `lambda_adv` with `alpha_sens = 0.0`.
Later, for an alpha sweep, set `sweep_x_key = "alpha_sens"` and keep a single lambda value fixed.
Change `sweep_name` if you change the training settings.

In [ ]:
def make_sweep_run_name(sweep_name, lambda_adv, alpha_sens):
    if lambda_adv == 0.0 and alpha_sens == 0.0:
        return f"{sweep_name}_baseline"
    run_name = f"{sweep_name}_lam_{lambda_adv:.2f}"
    if alpha_sens != 0.0:
        run_name += f"_alpha_{alpha_sens:.2f}"
    return run_name


def make_sweep_label(lambda_adv, alpha_sens):
    if lambda_adv == 0.0 and alpha_sens == 0.0:
        return "baseline"
    label = f"lam={lambda_adv:.1f}"
    if alpha_sens != 0.0:
        label += f", alpha={alpha_sens:.1f}"
    return label


sweep_name = "lambda_sweep_e5_lr1e-4"
sweep_x_key = "lambda_adv"
sweep_x_label = "lambda"

sweep_lambda_values = [0.0, 0.1, 0.2, 0.4, 0.8, 1.0]
sweep_alpha_values = [0.0]

# Current 5-epoch runs look largely settled by epochs 4-5, and 1e-4 has been stable.
sweep_epochs = 5
sweep_learning_rate = config.learning_rate
sweep_min_history = config.min_history
sweep_max_history = config.max_history
sweep_non_sensitive_only = config.non_sensitive_history_only

# Keep the sweep notebook-friendly on CPU. Set these to None on CUDA for full runs.
sweep_max_train_batches = 60 if device.type == "cpu" else None
sweep_max_test_batches = 30 if device.type == "cpu" else None
sweep_force_retrain = False

if sweep_x_key == "lambda_adv" and len(sweep_alpha_values) != 1:
    raise ValueError("Keep alpha fixed when sweeping lambda.")
if sweep_x_key == "alpha_sens" and len(sweep_lambda_values) != 1:
    raise ValueError("Keep lambda fixed when sweeping alpha.")

sweep_bundles = {}
sweep_history_by_label = {}

for alpha_sens in sweep_alpha_values:
    for lambda_adv in sweep_lambda_values:
        run_name = make_sweep_run_name(sweep_name, lambda_adv, alpha_sens)
        label = make_sweep_label(lambda_adv, alpha_sens)
        print(f"Loading/training {label} -> {run_name}")
        sweep_bundles[label] = load_or_train_bundle(
            run_name=run_name,
            lambda_adv=lambda_adv,
            alpha_sens=alpha_sens,
            min_history=sweep_min_history,
            max_history=sweep_max_history,
            non_sensitive_only=sweep_non_sensitive_only,
            epochs=sweep_epochs,
            learning_rate=sweep_learning_rate,
            max_train_batches=sweep_max_train_batches,
            max_test_batches=sweep_max_test_batches,
            force_retrain=sweep_force_retrain,
        )
        sweep_history_by_label[label] = load_run_history(run_name)

In [ ]:
sweep_curves_fig = plot_training_curves(
    history_by_run=sweep_history_by_label,
    output_path=paths.figures_root / f"{sweep_name}_{sweep_x_key}_curves.png",
)

sweep_summary_fig = plot_hparam_sweep_summary(
    history_by_run=sweep_history_by_label,
    output_path=paths.figures_root / f"{sweep_name}_{sweep_x_key}_tradeoff.png",
    hparam_name=sweep_x_key,
    hparam_label=sweep_x_label,
)

sweep_summary_rows = summarize_hparam_sweep(
    history_by_run=sweep_history_by_label,
    hparam_name=sweep_x_key,
)
sweep_summary_path = paths.runs_root / f"{sweep_name}_{sweep_x_key}_summary.json"
with open(sweep_summary_path, "w", encoding="utf-8") as handle:
    json.dump(sweep_summary_rows, handle, indent=2)

print(f"Saved training-curve figure: {sweep_curves_fig}")
print(f"Saved tradeoff figure: {sweep_summary_fig}")
print(f"Saved sweep summary: {sweep_summary_path}")

sweep_summary_rows